# Exploratory 50-record pilot
Run on an L4 or another native-BF16 GPU. This notebook downloads code from GitHub at a fixed commit; upload the private candidate JSONL separately. No dataset or outputs are embedded. Source qualification remains pending.


In [ ]:
import subprocess, sys
from pathlib import Path
REPO = Path("/content/algoverse-pilot-code")
REVISION = '2de6ae2b78b65eb12b017ef3642a0d978e6794f0'
subprocess.run(["git", "clone", "https://github.com/Dharshan2004/algoverse.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", REVISION], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == REVISION
print("Runner revision:", REVISION)
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==5.19.0", "mistral-common==1.12.0", "accelerate==1.15.0", "bitsandbytes==0.50.2", "huggingface_hub==1.33.0", "sentencepiece", "numpy==2.5.4"], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
print(torch.cuda.get_device_name(0))
uploaded = files.upload()
assert len(uploaded) == 1, "Upload the original candidate_inputs.jsonl only"
source = Path('/content/candidate_inputs.jsonl')
source.write_bytes(next(iter(uploaded.values())))


In [ ]:
import subprocess, sys
repo = Path('/content/algoverse-pilot-code')
run = Path('/content/pilot-run')
subprocess.run([sys.executable, '-u', str(repo/'pilot.py'), '--input', str(source), '--out', str(run)], check=True)
for model in ['ministral','gemma']:
    subprocess.run([sys.executable, str(repo/'pilot_report.py'), str(run/model), '--input', str(source), '--check-states', '--out', str(run/'reports'/model)], check=True)
(run/'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
import shutil
files.download(shutil.make_archive(str(run), 'zip', run))
